# PaddleOCR-VL — Extract 700 ảnh test
Chạy notebook này TRƯỚC để extract data, lưu JSON, rồi load vào notebook Vintern eval.

In [ ]:
# Cell 1: Cài thư viện
!pip install -q transformers==4.47.* --force-reinstall
!pip install -q numpy==1.26.4 --force-reinstall
!pip install -q Pillow tqdm

In [ ]:
# Cell 2: Restart kernel sau khi cài (chạy cell này rồi restart)
import os
os.kill(os.getpid(), 9)

In [ ]:
# Cell 3: Config đường dẫn
from pathlib import Path

PADDLE_MODEL_PATH = Path("/kaggle/input/datasets/maituananh511/paddleocr-vl-finetuned/paddleocr_vl_finetuned")  # ← sửa path model paddle
TEST_DATASET_PATH = "/kaggle/input/datasets/maituananh511/test-dataset-chart-vqa/vi_chart_dataset"
VIETNAMESE_DATA_PATH = Path("/kaggle/input/datasets/maituananh511/data-vietnamese/Data Vietnamese")
VIETNAMESE_IMAGES_PATH = VIETNAMESE_DATA_PATH / "images"
VIETNAMESE_JSONL_PATH  = VIETNAMESE_DATA_PATH / "viet_chart_vqa.jsonl"
OUTPUT_JSON = "/kaggle/working/paddle_extracted_700.json"

print("PADDLE_MODEL_PATH exists:", PADDLE_MODEL_PATH.exists())
print("TEST_DATASET_PATH exists:", Path(TEST_DATASET_PATH).exists())

In [ ]:
# Cell 4: Load PaddleOCR-VL model
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading PaddleOCR-VL...")
model = AutoModelForImageTextToText.from_pretrained(
    str(PADDLE_MODEL_PATH),
    torch_dtype=torch.float16,
).to(device).eval()

processor = AutoProcessor.from_pretrained(str(PADDLE_MODEL_PATH))
print("✅ PaddleOCR-VL ready")

In [ ]:
# Cell 5: Hàm extract 1 ảnh
def extract_chart(image: Image.Image) -> str:
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": "Chart Recognition:"}
        ]
    }]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        images_kwargs={"size": {"shortest_edge": 560, "longest_edge": 1024 * 28 * 28}},
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256)

    result = processor.decode(
        outputs[0][inputs["input_ids"].shape[-1]:-1],
        skip_special_tokens=True
    )
    return result.strip()

In [ ]:
# Cell 6: Load 700 ảnh test từ dataset
import shutil, os, json
from datasets import load_from_disk, concatenate_datasets, Dataset
import pyarrow as pa

# Copy dataset sang working dir
SRC = TEST_DATASET_PATH
DST = "/kaggle/working/vi_chart_dataset"
if not os.path.exists(DST):
    shutil.copytree(SRC, DST)

vi_chart_dataset = load_from_disk(DST)
print(vi_chart_dataset)

# Load Vietnamese test data (200 ảnh)
vietnamese_records = []
with open(VIETNAMESE_JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            vietnamese_records.append(json.loads(line))
print(f"Vietnamese records: {len(vietnamese_records)}")

# Lấy 200 ảnh test Vietnamese (tương tự notebook train)
TEST_SIZE = 200
vn_test_records = vietnamese_records[-TEST_SIZE:]
print(f"Vietnamese test: {len(vn_test_records)} records")
print(f"VietChartVQA test: {len(vi_chart_dataset['test'])} records")
print(f"Total test: {len(vi_chart_dataset['test']) + len(vn_test_records)} records")

In [ ]:
# Cell 7: Extract toàn bộ 700 ảnh test → lưu JSON
from tqdm import tqdm

extracted = {}  # {image_id: extracted_text}

# Load existing nếu bị ngắt giữa chừng
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        extracted = json.load(f)
    print(f"Resumed: {len(extracted)} images already extracted")

# ── Part 1: VietChartVQA test (500 ảnh) ──────────────────────────────
print("\n=== Part 1: VietChartVQA test ===")
for i in tqdm(range(len(vi_chart_dataset['test'])), desc="VietChartVQA"):
    item = vi_chart_dataset['test'][i]
    img_id = str(item['id'])

    if img_id in extracted:
        continue  # skip nếu đã extract

    try:
        image = item['image']
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image).convert('RGB')
        extracted[img_id] = extract_chart(image)
    except Exception as e:
        print(f"Error {img_id}: {e}")
        extracted[img_id] = ""

    # Save sau mỗi 10 ảnh để tránh mất data
    if i % 10 == 0:
        with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
            json.dump(extracted, f, ensure_ascii=False, indent=2)

# ── Part 2: Vietnamese test (200 ảnh) ────────────────────────────────
print("\n=== Part 2: Vietnamese test ===")
for i, record in enumerate(tqdm(vn_test_records, desc="Vietnamese")):
    img_id = record['id']

    if img_id in extracted:
        continue

    try:
        img_path = VIETNAMESE_IMAGES_PATH / record['image']
        image = Image.open(img_path).convert('RGB')
        extracted[img_id] = extract_chart(image)
    except Exception as e:
        print(f"Error {img_id}: {e}")
        extracted[img_id] = ""

    if i % 10 == 0:
        with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
            json.dump(extracted, f, ensure_ascii=False, indent=2)

# Save final
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(extracted, f, ensure_ascii=False, indent=2)

print(f"\n✅ Done! Extracted {len(extracted)} images")
print(f"Saved to: {OUTPUT_JSON}")

In [ ]:
# Cell 8: Kiểm tra kết quả
with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Total extracted: {len(data)}")
print("\nSample outputs:")
for i, (k, v) in enumerate(list(data.items())[:3]):
    print(f"\n[{k}]: {v[:200]}...")
    
empty_count = sum(1 for v in data.values() if not v.strip())
print(f"\nEmpty extractions: {empty_count}/{len(data)}")

In [ ]:
# Cell 9: Zip file JSON lại
import zipfile, os

ZIP_PATH = "/kaggle/working/paddle_extracted_700.zip"

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUTPUT_JSON, "paddle_extracted_700.json")

size_mb = os.path.getsize(ZIP_PATH) / (1024*1024)
print(f"✅ Zipped: {ZIP_PATH} ({size_mb:.2f} MB)")
print("Download file này rồi upload vào notebook Vintern eval")